# Election Bloc Change Prediction Project
## Notebook 05 — Historical rolling delta modeling

Compares persistence with Ridge delta models across multiple expanding-window temporal backtests. The model never receives locality identity. K24→K25 remains locked.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 300)
REPO_URL = "https://github.com/IfatDav/Election_Bloc_Prediction_Project.git"
DEFAULT_REPO = Path("/content/Election_Bloc_Prediction_Project")

def locate_repo():
    candidates=[]
    if os.getenv("ELECTION_PROJECT_ROOT"):
        candidates.append(Path(os.environ["ELECTION_PROJECT_ROOT"]).expanduser())
    cwd=Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, DEFAULT_REPO]
    for p in candidates:
        if (p/"data"/"raw").exists(): return p
    if Path("/content").exists():
        if DEFAULT_REPO.exists(): shutil.rmtree(DEFAULT_REPO)
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(DEFAULT_REPO)],check=True)
        return DEFAULT_REPO
    raise FileNotFoundError("Repository not found. Set ELECTION_PROJECT_ROOT.")

REPO_ROOT=locate_repo()
RAW_DIR=REPO_ROOT/"data"/"raw"
INTERIM_DIR=REPO_ROOT/"data"/"interim"
PROCESSED_DIR=REPO_ROOT/"data"/"processed"
REPORTS_DIR=REPO_ROOT/"reports"
TABLES_DIR=REPORTS_DIR/"tables"
FIGURES_DIR=REPORTS_DIR/"figures"
SUMMARIES_DIR=REPORTS_DIR/"summaries"
MODELS_DIR=REPO_ROOT/"models"
for d in [INTERIM_DIR,PROCESSED_DIR,TABLES_DIR,FIGURES_DIR,SUMMARIES_DIR,MODELS_DIR]: d.mkdir(parents=True,exist_ok=True)
print("Repository:",REPO_ROOT)

In [ ]:
import joblib, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
DATA=INTERIM_DIR/"modeling_transition_features.csv"; MANIFEST=SUMMARIES_DIR/"feature_manifest.json"
if not DATA.exists() and (REPO_ROOT/".git").exists(): subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin","main"],check=True)
df=pd.read_csv(DATA,dtype={"locality_symbol":"string"},low_memory=False); manifest=json.loads(MANIFEST.read_text(encoding="utf-8-sig"))
MODELED_BLOCS=["Right","Center_Left","Haredi","Arab"]; TARGET=[f"delta_clr_{b}" for b in MODELED_BLOCS]; PREVCLR=[f"previous_clr_{b}" for b in MODELED_BLOCS]; PREVPCT=[f"previous_{b}_pct" for b in MODELED_BLOCS]; CURRPCT=[f"current_{b}_pct" for b in MODELED_BLOCS]
ordered=[f"K{n}_to_K{n+1}" for n in range(16,25)]; DEV=ordered[:-1]; FOLDS=DEV[3:]


def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (series.astype("string").str.strip().str.lower()
            .map({"true":True,"false":False,"1":True,"0":False,"yes":True,"no":False})
            .fillna(False).astype(bool))

df["has_current_core_features"] = to_bool(df["has_current_core_features"])

print("Rows",len(df),"development folds",FOLDS)

## Utilities

In [ ]:
def onehot():
    try:return OneHotEncoder(handle_unknown="ignore",sparse_output=False)
    except TypeError:return OneHotEncoder(handle_unknown="ignore",sparse=False)
def filter_features(data,cols,threshold=.35):
    keep=[]
    for c in cols:
        if c not in data: continue
        if c in {"locality_type","analysis_group"}:
            if data[c].nunique(dropna=True)>1:keep.append(c)
        else:
            x=pd.to_numeric(data[c],errors="coerce")
            if x.isna().mean()<=threshold and x.nunique(dropna=True)>1:keep.append(c)
    return keep
def prep(data,cols):
    x=data[cols].copy()
    for c in cols:
        if c in {"locality_type","analysis_group"}:x[c]=x[c].astype("string").fillna("Unknown").astype(str)
        else:x[c]=pd.to_numeric(x[c],errors="coerce")
    return x
def pipe(cols,alpha):
    cat=[c for c in cols if c in {"locality_type","analysis_group"}];num=[c for c in cols if c not in cat];trs=[]
    if num:trs.append(("num",Pipeline([("imp",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("scale",StandardScaler())]),num))
    if cat:trs.append(("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",onehot())]),cat))
    return Pipeline([("pre",ColumnTransformer(trs)),("model",Ridge(alpha=alpha))])
def center(z):z=np.asarray(z,float);return z-z.mean(1,keepdims=True)
def invclr(z):e=np.exp(z-z.max(1,keepdims=True));return e/e.sum(1,keepdims=True)*100
def to_pct(data,delta):return invclr(data[PREVCLR].to_numpy(float)+center(delta))
def score(data,pred):
    y=data[CURRPCT].to_numpy(float);ae=np.abs(y-pred);row=ae.mean(1);w=pd.to_numeric(data.current_valid_votes,errors="coerce").fillna(0).to_numpy();return float(ae.mean()),float(np.average(row,weights=w)) if w.sum()>0 else np.nan,float(np.median(row))

## Candidate sets and expanding-window backtest

In [ ]:
sets={
"Ridge — history only":manifest["feature_sets"]["history_only"],
"Ridge — history + levels":manifest["feature_sets"]["history_plus_levels"],
"Ridge — history + levels + change":manifest["feature_sets"]["history_plus_levels_and_change"],
}
ALPHAS=[10,100,1000,3000,10000,30000]
rows=[]; group_rows=[]; bloc_rows=[]
for fold in FOLDS:
    idx=ordered.index(fold); train_ids=ordered[:idx]
    tr=df[df.transition_id.isin(train_ids)&df.has_current_core_features.astype(bool)].copy(); te=df[df.transition_id.eq(fold)&df.has_current_core_features.astype(bool)].copy()
    if len(te)==0: continue
    pers=te[PREVPCT].to_numpy(float); mae,wmae,med=score(te,pers);rows.append({"transition_id":fold,"model":"Previous-election persistence","alpha":0,"rows":len(te),"mae":mae,"weighted_mae":wmae,"median_row_mae":med})
    for name,rawcols in sets.items():
        cols=filter_features(tr,rawcols)
        # Alpha is chosen by prior-transition average error inside this training window.
        inner=[]
        inner_folds=[t for t in train_ids[2:] if t in tr.transition_id.unique()]
        for a in ALPHAS:
            vals=[]
            for inner_fold in inner_folds:
                j=ordered.index(inner_fold);itr=tr[tr.transition_id.isin(ordered[:j])];iva=tr[tr.transition_id.eq(inner_fold)]
                if len(itr)<50 or len(iva)==0:continue
                icols=filter_features(itr,rawcols);m=pipe(icols,a).fit(prep(itr,icols),itr[TARGET].to_numpy(float));vals.append(score(iva,to_pct(iva,m.predict(prep(iva,icols))))[0])
            inner.append((np.mean(vals) if vals else np.inf,a))
        alpha=min(inner)[1];m=pipe(cols,alpha).fit(prep(tr,cols),tr[TARGET].to_numpy(float));pred=to_pct(te,m.predict(prep(te,cols)));mae,wmae,med=score(te,pred)
        rows.append({"transition_id":fold,"model":name,"alpha":alpha,"rows":len(te),"mae":mae,"weighted_mae":wmae,"median_row_mae":med,"feature_count":len(cols)})
        y=te[CURRPCT].to_numpy(float)
        for g,sub in te.groupby("analysis_group"):
            loc=te.index.get_indexer(sub.index);gm,gw,gmed=score(sub,pred[loc]);group_rows.append({"transition_id":fold,"model":name,"analysis_group":g,"rows":len(sub),"mae":gm,"weighted_mae":gw,"median_row_mae":gmed})
        for j,b in enumerate(MODELED_BLOCS):bloc_rows.append({"transition_id":fold,"model":name,"bloc":b,"rows":len(te),"mae":float(np.abs(y[:,j]-pred[:,j]).mean())})
comparison=pd.DataFrame(rows); summary=comparison.groupby("model").agg(mean_mae=("mae","mean"),std_mae=("mae","std"),mean_weighted_mae=("weighted_mae","mean"),folds=("transition_id","nunique")).sort_values("mean_mae")
summary

## Select and save the learned architecture

In [ ]:
learned=summary.drop(index="Previous-election persistence",errors="ignore");best_name=learned.index[0]
alpha_table=comparison[comparison.model.eq(best_name)].groupby("alpha").mae.mean();best_alpha=float(alpha_table.idxmin())
dev=df[df.transition_id.isin(DEV)&df.has_current_core_features.astype(bool)].copy();cols=filter_features(dev,sets[best_name]);best_model=pipe(cols,best_alpha).fit(prep(dev,cols),dev[TARGET].to_numpy(float))
artifact={"model_name":best_name,"alpha":best_alpha,"feature_columns":cols,"pipeline":best_model,"training_transitions":DEV,"target_columns":TARGET,"bloc_order":MODELED_BLOCS}
paths={"comparison":TABLES_DIR/"historical_temporal_model_comparison.csv","summary_table":TABLES_DIR/"historical_temporal_model_summary.csv","group":TABLES_DIR/"historical_temporal_model_by_group.csv","bloc":TABLES_DIR/"historical_temporal_model_by_bloc.csv","model":MODELS_DIR/"best_historical_delta_model.joblib","selection":SUMMARIES_DIR/"selected_historical_delta_model.json","summary":SUMMARIES_DIR/"notebook_05_summary.json"}
comparison.to_csv(paths["comparison"],index=False,encoding="utf-8-sig");summary.reset_index().to_csv(paths["summary_table"],index=False,encoding="utf-8-sig");pd.DataFrame(group_rows).to_csv(paths["group"],index=False,encoding="utf-8-sig");pd.DataFrame(bloc_rows).to_csv(paths["bloc"],index=False,encoding="utf-8-sig");joblib.dump(artifact,paths["model"])
selection={"best_learned_model":best_name,"alpha":best_alpha,"mean_temporal_mae":float(summary.loc[best_name,"mean_mae"]),"persistence_mean_temporal_mae":float(summary.loc["Previous-election persistence","mean_mae"]),"learned_model_beats_persistence":bool(summary.loc[best_name,"mean_mae"]<summary.loc["Previous-election persistence","mean_mae"]),"final_test_status":"K24_to_K25 locked"}
paths["selection"].write_text(json.dumps(selection,ensure_ascii=False,indent=2),encoding="utf-8");paths["summary"].write_text(json.dumps({"notebook":"05_delta_modeling","temporal_folds":FOLDS,"selection":selection},ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(selection,ensure_ascii=False,indent=2))